# 🌊 Lagoon Water Quality Monitor — Data Update

This notebook exports FAI and NDCI remote sensing data from Google Earth Engine for the lagoon monitoring dashboard.

**How to use:**
1. Set the update mode below (full or incremental)
2. Click **Runtime → Run all**
3. Authorise your Google account when prompted
4. Wait for processing to complete
5. Download the generated zip file from the link at the bottom
6. Extract and copy the `data/` folder into your dashboard folder, replacing existing files
7. Open `index.html` to view the updated dashboard

## ⚙️ Configuration

Set your update mode here. Change `LAST_LOCAL_DATE` to the latest date shown in your dashboard (top-right corner).

In [ ]:
# ============================================================
# UPDATE MODE
# ============================================================
# For first-time full export: set LAST_LOCAL_DATE = None
# For incremental update: set LAST_LOCAL_DATE to your dashboard's latest date
#   (shown in the top-right corner of the dashboard)

LAST_LOCAL_DATE = None  # e.g. "2026-08-27" for incremental, or None for full export

# ============================================================
# PROJECT SETTINGS (do not change)
# ============================================================
GEE_PROJECT = "lagoon-dashboard-507406"
SATELLITE = "COPERNICUS/S2_SR_HARMONIZED"
CLOUD_MAX = 30
DATE_START = "2024-01-01"
SCALE = 10

FAI_VIS = {
    "min": -0.05, "max": 0.15,
    "palette": ["blue", "cyan", "green", "yellow", "orange", "red"]
}
NDCI_VIS = {
    "min": -0.1, "max": 0.5,
    "palette": ["blue", "cyan", "green", "yellow", "orange", "red"]
}

# Lagoon boundary GeoJSON URL (loaded from GitHub)
GEOJSON_URL = "https://raw.githubusercontent.com/Cathy327/lagoon-monitor/main/lagoon.geojson"

## 📦 Setup

Install dependencies and authenticate with Google Earth Engine.

In [ ]:
# Install dependencies
!pip install -q earthengine-api rasterio Pillow pandas

import ee
import json
import os
import time
import datetime
import requests
import numpy as np
import rasterio
import pandas as pd
from PIL import Image as PILImage
from pathlib import Path
from io import BytesIO
import shutil

# Authenticate & initialise GEE
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print("✓ GEE initialised successfully")


## 🗺️ Load Lagoon Boundary

In [ ]:
# Load lagoon boundary from GitHub
try:
    geojson_data = requests.get(GEOJSON_URL).json()
    lakes = ee.FeatureCollection(geojson_data)
    print("✓ Lagoon boundary loaded from GitHub")
except Exception:
    # Fallback: hardcoded coordinates
    print("⚠ Could not load from GitHub, using hardcoded coordinates")
    coords = [[143.84683079111198,-37.60949743581094],[143.8469017735482,-37.609491975623534],[143.8470055171089,-37.60950835618575],[143.84712018104437,-37.60953565712276],[143.84726214591686,-37.60957933862199],[143.8474860136004,-37.60964486087084],[143.84757883678625,-37.60969400255747],[143.8476279784729,-37.60978682574333],[143.8476279784729,-37.6099014896788],[143.84759521734847,-37.60999977305206],[143.8475242349122,-37.610108976800134],[143.84739319041455,-37.61019087961119],[143.84720754404282,-37.61033284448368],[143.8470546587955,-37.61047480935617],[143.8469836763593,-37.61058401310424],[143.84692361429785,-37.61075327891375],[143.84684171148677,-37.61094984566027],[143.84683079111198,-37.61108635034536],[143.84683079111198,-37.61123377540526],[143.84684171148677,-37.61132113840372],[143.8468471716742,-37.61142488196438],[143.8468744726112,-37.611506784775436],[143.8468744726112,-37.61156138664947],[143.8468799327986,-37.61163236908571],[143.8468526318616,-37.611665130210135],[143.84681441054977,-37.61168151077234],[143.84672704755133,-37.611692431147155],[143.84660692342842,-37.61170881170936],[143.84650317986777,-37.611757953395994],[143.84639397611969,-37.61185077658185],[143.84633937424567,-37.61194359976771],[143.84626293162202,-37.61210740538982],[143.8461209667495,-37.61226029063712],[143.84602814356367,-37.61229851194894],[143.84585887775415,-37.61234219344817],[143.84574421381868,-37.61232035269856],[143.84566231100763,-37.61229305176154],[143.84554764707215,-37.612238449887506],[143.8454657442611,-37.61215108688905],[143.84540568219967,-37.6120801044528],[143.84538384145006,-37.61196544051733],[143.84539476182485,-37.61188899789368],[143.84542752294928,-37.61181255527003],[143.84545482388629,-37.61175249320859],[143.84553126650994,-37.61168151077234],[143.84557494800916,-37.611621448710906],[143.84566777119502,-37.611495864400624],[143.84570053231946,-37.61146856346361],[143.84586979812897,-37.61122285503045],[143.8459462407526,-37.61111365128238],[143.84602268337625,-37.6109935271595],[143.8460936658125,-37.61081880116259],[143.84614826768654,-37.61057309272943],[143.84620286956059,-37.61026732223483],[143.8462520112472,-37.610081675863114],[143.84630661312124,-37.609945171178026],[143.84636121499528,-37.60986872855438],[143.8464321974315,-37.60977044518111],[143.8465687021166,-37.60965032105824],[143.84665606511507,-37.60957933862199],[143.84678164942534,-37.60951381637315],[143.84683079111198,-37.60949743581094]]
    geom = ee.Geometry.Polygon([coords])
    lakes = ee.FeatureCollection([ee.Feature(geom)])

# Get bounds for export
bounds = lakes.geometry().bounds()
bounds_coords = bounds.coordinates().getInfo()[0]
bounds_info = {
    "southwest": [bounds_coords[0][0], bounds_coords[0][1]],
    "northeast": [bounds_coords[2][0], bounds_coords[2][1]]
}
print(f"  Bounds: SW {bounds_info['southwest']}, NE {bounds_info['northeast']}")


## 🛰️ Query Satellite Data

In [ ]:
# Determine query range
if LAST_LOCAL_DATE is None:
    query_start = DATE_START
    mode = "full"
    print("Mode: FULL EXPORT")
else:
    from datetime import date, timedelta
    last = date.fromisoformat(LAST_LOCAL_DATE)
    query_start = (last + timedelta(days=1)).isoformat()
    mode = "incremental"
    print(f"Mode: INCREMENTAL (from {query_start})")

end_date = datetime.date.today().isoformat()
print(f"Query range: {query_start} to {end_date}")

# Load Sentinel-2
s2 = (ee.ImageCollection(SATELLITE)
    .filterBounds(lakes)
    .filterDate(query_start, end_date)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_MAX))
    .sort("system:time_start"))

s2 = s2.map(lambda img: img.set("date", img.date().format("YYYY-MM-dd")))
date_list = s2.aggregate_array("date").distinct().sort().getInfo()

print(f"\nFound {len(date_list)} new dates")
if not date_list:
    print("No new images available. Your dashboard is up to date.")
    print("(This may be due to cloud cover in recent acquisitions)")
else:
    for d in date_list:
        print(f"  - {d}")


## 📊 Compute Indices & Export

In [ ]:
def compute_indices(image):
    """Add FAI and NDCI bands to a Sentinel-2 image."""
    red = image.select("B4").multiply(0.0001)
    nir = image.select("B8").multiply(0.0001)
    swir = image.select("B11").multiply(0.0001)
    fai = nir.subtract(
        red.add(swir.subtract(red).multiply((842 - 665) / (1610 - 665)))
    ).rename("FAI")
    ndci = image.normalizedDifference(["B5", "B4"]).rename("NDCI")
    return image.addBands(fai).addBands(ndci)


def download_image(image, index_name, vis_params, lakes, bounds, scale, output_path):
    """Download a rendered index image as GeoTIFF, convert to PNG."""
    idx_image = image.select(index_name).clip(lakes)
    viz = idx_image.visualize(**vis_params)
    mask = ee.Image.constant(255).toByte().clip(lakes).unmask(0)
    rgba = viz.addBands(mask.rename("alpha"))

    url = rgba.getDownloadURL({
        "region": bounds,
        "scale": scale,
        "format": "GEO_TIFF",
        "bands": ["vis-red", "vis-green", "vis-blue", "alpha"],
    })

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    # Convert GeoTIFF bytes to PNG
    with rasterio.open(BytesIO(response.content)) as src:
        r, g, b, a = src.read(1), src.read(2), src.read(3), src.read(4)
        rgba_arr = np.stack([r, g, b, a], axis=-1).astype(np.uint8)
        PILImage.fromarray(rgba_arr, mode="RGBA").save(output_path, "PNG")


# Create output directories
OUTPUT_DIR = Path("lagoon_data")
FAI_DIR = OUTPUT_DIR / "images" / "fai"
NDCI_DIR = OUTPUT_DIR / "images" / "ndci"
FAI_DIR.mkdir(parents=True, exist_ok=True)
NDCI_DIR.mkdir(parents=True, exist_ok=True)

# Export rendered images
if date_list:
    total = len(date_list) * 2
    count = 0
    failed = []

    for date_str in date_list:
        day_images = s2.filter(ee.Filter.eq("date", date_str))
        composite = day_images.map(compute_indices).mean()

        for idx_name, vis, out_dir, prefix in [
            ("FAI", FAI_VIS, FAI_DIR, "fai"),
            ("NDCI", NDCI_VIS, NDCI_DIR, "ndci"),
        ]:
            count += 1
            output_path = out_dir / f"{prefix}_{date_str}.png"

            for attempt in range(3):
                try:
                    download_image(composite, idx_name, vis, lakes, bounds, SCALE, output_path)
                    print(f"  [{count}/{total}] {prefix}_{date_str}  ✓")
                    break
                except Exception as e:
                    if attempt < 2:
                        print(f"  [{count}/{total}] {prefix}_{date_str}  ✗ retry ({e})")
                        time.sleep(30)
                    else:
                        print(f"  [{count}/{total}] {prefix}_{date_str}  ✗ FAILED")
                        failed.append(f"{prefix}_{date_str}")

            time.sleep(2)  # Rate limiting

    if failed:
        print(f"\n⚠ {len(failed)} images failed: {failed}")
    else:
        print(f"\n✓ All {total} images exported successfully")
else:
    print("No images to export.")


## 📈 Generate Time Series CSV

In [ ]:
if date_list:
    # For full mode, query ALL dates for CSV
    if mode == "full":
        s2_all = (ee.ImageCollection(SATELLITE)
            .filterBounds(lakes)
            .filterDate(DATE_START, end_date)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_MAX))
            .sort("system:time_start")
            .map(lambda img: img.set("date", img.date().format("YYYY-MM-dd"))))
        csv_dates = s2_all.aggregate_array("date").distinct().sort().getInfo()
        s2_for_csv = s2_all
    else:
        csv_dates = date_list
        s2_for_csv = s2

    print(f"Computing statistics for {len(csv_dates)} dates...")
    rows = []

    for i, date_str in enumerate(csv_dates):
        day_images = s2_for_csv.filter(ee.Filter.eq("date", date_str))
        composite = day_images.map(compute_indices).mean()

        stats = composite.select(["FAI", "NDCI"]).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), "", True),
            geometry=lakes.geometry(),
            scale=20,
            maxPixels=1e9,
        ).getInfo()

        rows.append({
            "date": date_str,
            "FAI_mean": stats.get("FAI_mean"),
            "FAI_stdDev": stats.get("FAI_stdDev"),
            "NDCI_mean": stats.get("NDCI_mean"),
            "NDCI_stdDev": stats.get("NDCI_stdDev"),
        })

        if (i + 1) % 10 == 0 or i == len(csv_dates) - 1:
            print(f"  {i+1}/{len(csv_dates)} done")

    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset="date", keep="last").sort_values("date")
    csv_path = OUTPUT_DIR / "full_timeseries.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n✓ CSV saved: {len(df)} rows")
else:
    print("No new data to add to CSV.")


## 📋 Generate Metadata

In [ ]:
# Save bounds.json
with open(OUTPUT_DIR / "bounds.json", "w") as f:
    json.dump(bounds_info, f, indent=2)
print("✓ bounds.json saved")

# Save dates.json (all available dates from local images)
all_dates = sorted([f.stem.replace("fai_", "") for f in FAI_DIR.glob("fai_*.png")])
with open(OUTPUT_DIR / "dates.json", "w") as f:
    json.dump(all_dates, f, indent=2)
print(f"✓ dates.json saved: {len(all_dates)} dates")

# Summary
print(f"\n{"="*50}")
print(f"  Export complete!")
print(f"  Total images: {len(all_dates)} dates × 2 indices")
print(f"  Latest date: {all_dates[-1] if all_dates else 'N/A'}")
print(f"{"="*50}")


## 📥 Download Results

Run this cell to package all data into a zip file for download.

In [ ]:
# Package into zip
zip_name = "lagoon_dashboard_data"
shutil.make_archive(zip_name, "zip", ".", "lagoon_data")
zip_path = f"{zip_name}.zip"
size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"✓ Packaged: {zip_path} ({size_mb:.1f} MB)")

# Download link
try:
    from google.colab import files
    print("\nDownloading...")
    files.download(zip_path)
except ImportError:
    print(f"\nNot running in Colab. File saved at: {zip_path}")
